# 7. Membuat Model Prediksi (Frontend Streamlit)

In [ ]:
%%writefile app_streamlit.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import mysql.connector

# KONFIGURASI HALAMAN
st.set_page_config(page_title='Dashboard AI Kodim', page_icon='🪖', layout='wide')

# ================= CUSTOM CSS (UI PREMIUM) =====================
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&display=swap');
    html, body, [class*="css"]  {
        font-family: 'Inter', sans-serif;
    }
    
    .stButton > button {
        background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%);
        color: white;
        border-radius: 10px;
        border: none;
        transition: all 0.3s ease;
        font-weight: 600;
        padding: 0.6rem 1rem;
        box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);
    }
    
    .stButton > button:hover {
        transform: translateY(-2px);
        box-shadow: 0 10px 15px -3px rgba(0,0,0,0.1);
        color: white;
        border: none;
    }
    
    .title-banner {
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        padding: 2rem;
        border-radius: 15px;
        color: white;
        text-align: center;
        margin-bottom: 2rem;
        box-shadow: 0 10px 15px -3px rgba(0,0,0,0.1);
    }
    
    .stTextInput > div > div > input {
        border-radius: 8px;
    }
</style>
""", unsafe_allow_html=True)


# ================= DATABASE INIT =====================
@st.cache_resource
def get_db_connection():
    try:
        # Koneksi awal ke MySQL XAMPP tanpa memilih database
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password=""
        )
        cursor = conn.cursor()
        
        # Buat database jika belum ada
        cursor.execute("CREATE DATABASE IF NOT EXISTS db_kodim")
        cursor.execute("USE db_kodim")
        
        # Buat tabel Login
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INT AUTO_INCREMENT PRIMARY KEY,
                username VARCHAR(50) UNIQUE,
                password VARCHAR(100)
            )
        """)
        
        # Buat Default User jika tabel kosong
        cursor.execute("SELECT * FROM users WHERE username='admin'")
        if not cursor.fetchone():
            cursor.execute("INSERT INTO users (username, password) VALUES ('admin', 'admin123')")
            conn.commit()
            
        # Buat tabel Riwayat Prediksi (Sesuai Skema Flask Hari Kemarin!)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS riwayat_prediksi (
                id VARCHAR(100) PRIMARY KEY,
                nama VARCHAR(255),
                umur FLOAT,
                tb FLOAT,
                bb FLOAT,
                lari FLOAT,
                pullup FLOAT,
                situp FLOAT,
                pushup FLOAT,
                shuttle FLOAT,
                hasil VARCHAR(50)
            )
        """)
        conn.commit()
        return conn
    except Exception as e:
        st.error(f"🚨 Gagal koneksi ke Database MySQL XAMPP. Pastikan XAMPP sedang START! Error: {e}")
        st.stop()

# ================= LOGIN SYSTEM =====================
# Sistem Anti-Logout Waktu Layar Direfresh (F5)
if hasattr(st, "query_params"):
    if st.query_params.get("auth") == "true":
        st.session_state['logged_in'] = True
        if 'username' not in st.session_state:
            st.session_state['username'] = "admin (Pulih)"

if 'logged_in' not in st.session_state:
    st.session_state['logged_in'] = False

if not st.session_state['logged_in']:
    # Tampilan Halaman Login Premium
    st.markdown("""
        <div class='title-banner'>
            <h1 style='color: white; margin-bottom:0;'>🔒 Markas Prediksi Personil Kodim</h1>
            <p style='color: #cbd5e1; font-size:1.1rem; margin-top:0;'>Masukkan otorita kemanan mesin buatan (SVM)</p>
        </div>
    """, unsafe_allow_html=True)
    
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        with st.form("login_form"):
            st.markdown("<h3 style='text-align: center;'>Identifikasi Pengguna</h3>", unsafe_allow_html=True)
            username = st.text_input("Username", placeholder="👤 Silakan Ketik Username Akses", label_visibility="collapsed")
            password = st.text_input("Password", placeholder="🔑 Silakan Ketik Kata Sandi Rahasia", type="password", label_visibility="collapsed")
            st.write("") # spacer
            submit = st.form_submit_button("Masuk ke Sistem Utama 🚀", use_container_width=True)
            
            if submit:
                conn = get_db_connection()
                if conn:
                    cursor = conn.cursor()
                    cursor.execute("SELECT * FROM users WHERE username=%s AND password=%s", (username, password))
                    user = cursor.fetchone()
                    
                    if user:
                        st.session_state['logged_in'] = True
                        st.session_state['username'] = username
                        if hasattr(st, "query_params"):
                            st.query_params["auth"] = "true"
                        st.rerun()
                    else:
                        st.error("❌ Username atau Password Salah!")
else:
    # ================= APLIKASI UTAMA (DASHBOARD) =====================
    conn = get_db_connection()
    
    # Header Atas (Bisa Logout)
    col1, col2 = st.columns([8, 2])
    with col1:
        st.title('🪖 Mesin Kecerdasan Buatan (SVM) - Uji Semapta')
        st.write(f"Selamat datang Komandan **{st.session_state['username']}**. Sistem Database MySQL [🟢 AKTIF].")
    with col2:
        st.write("") # Spacer
        if st.button("🚪 Logout (Keluar)", use_container_width=True):
            st.session_state['logged_in'] = False
            if hasattr(st, "query_params"):
                st.query_params.clear()
            st.rerun()
            
    # Load Model
    try:
        model = joblib.load('model_svm.pkl')
        scaler = joblib.load('scaler.pkl')
    except Exception as e:
        st.error("Model SVM gagal dimuat. Harap jalankan file Model Training terlebih dahulu.")
        st.stop()

    # Form Sidebar (Kiri)
    st.sidebar.header('📝 Identitas & Nilai Fisik')
    
    id_prajurit = st.sidebar.text_input("No. Siswa / ID Prajurit", placeholder="Ketik kombinasi Huruf/Angka (Cth: PR-01)")
    nama_prajurit = st.sidebar.text_input("Nama Lengkap", placeholder="Ketik Tulisan (Cth: Budi Santoso)")
    
    st.sidebar.markdown("---")
    umur = st.sidebar.number_input('Umur (Tahun)', min_value=17, max_value=60, value=None, placeholder="Cth: 25")
    tb = st.sidebar.number_input('Tinggi Badan (cm)', min_value=140, max_value=200, value=None, placeholder="Cth: 175")
    bb = st.sidebar.number_input('Berat Badan (kg)', min_value=40, max_value=120, value=None, placeholder="Cth: 65")
    lari = st.sidebar.number_input('Lari 12 Menit (Jarak Meter)', value=None, step=10.0, placeholder="Cth: 2400")
    pullup = st.sidebar.number_input('Pull Up (Jumlah)', value=None, placeholder="Cth: 12")
    situp = st.sidebar.number_input('Sit Up (Jumlah)', value=None, placeholder="Cth: 40")
    pushup = st.sidebar.number_input('Push Up (Jumlah)', value=None, placeholder="Cth: 35")
    shuttle = st.sidebar.number_input('Shuttle Run (Detik)', value=None, placeholder="Cth: 16.5")

    # Eksekusi (Tombol Utama)
    if st.sidebar.button('🧠 Analisa & Simpan ke Database', use_container_width=True, type='primary'):
        if not id_prajurit or not nama_prajurit:
            st.sidebar.error("⚠️ Lengkapi No. Siswa/ID dan Nama Prajurit terlebih dahulu!")
        elif None in [umur, tb, bb, lari, pullup, situp, pushup, shuttle]:
            st.sidebar.error("⚠️ Lengkapi kedelapan angka Fisik sebelum menekan tombol!")
        else:
            # Hitung SVM (Mesin Pintar)
            data_fisik = np.array([[umur, tb, bb, lari, pullup, situp, pushup, shuttle]])
            data_scaled = scaler.transform(data_fisik)
            hasil_akhir = model.predict(data_scaled)[0]
            
            # Simpan ke MySQL XAMPP
            try:
                cursor = conn.cursor()
                cursor.execute("SELECT id FROM riwayat_prediksi WHERE id = %s", (id_prajurit,))
                if cursor.fetchone():
                    st.error(f"⚠️ Gagal Disimpan! ID Prajurit **{id_prajurit}** sudah pernah dimasukkan sebelumnya.")
                else:
                    query = """
                        INSERT INTO riwayat_prediksi (id, nama, umur, tb, bb, lari, pullup, situp, pushup, shuttle, hasil)
                        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    """
                    values = (id_prajurit, nama_prajurit, umur, tb, bb, lari, pullup, situp, pushup, shuttle, hasil_akhir)
                    cursor.execute(query, values)
                    conn.commit()
                    
                    st.success(f'### Kategori Kelulusan Ditetapkan (Tingkat Prediksi): **{hasil_akhir}**')
                    st.info(f"💾 Data **{nama_prajurit}** ({id_prajurit}) berhasil diamankan ke dalam Bunker Database MySQL.")
                    if hasil_akhir == "TL":
                        st.snow()
                    else:
                        st.balloons()
            except Exception as e:
                st.error(f"Terjadi kesalahan saat injeksi ke SQL Database: {e}")
                
    st.markdown("---")
    st.subheader("🗄️ Tabel Catatan Prajurit (Live MySQL Sync)")
    
    # Ambil Ulang Data dari Tabel Database
    try:
        query_tabel = ("SELECT id as 'ID Prajurit', nama as 'Nama', umur as 'Umur', "
                       "tb as 'TB', bb as 'BB', lari as 'Lari', pullup as 'Pull-Up', "
                       "situp as 'Sit-Up', pushup as 'Push-Up', shuttle as 'Shuttle Run', "
                       "hasil as 'Status Akhir' FROM riwayat_prediksi ORDER BY id DESC")
        df_riwayat = pd.read_sql_query(query_tabel, conn)
        
        if df_riwayat.empty:
            st.write("*(Belum ada data historis uji coba terbaru yang tersimpan di MySQL).*")
        else:
            st.dataframe(df_riwayat, use_container_width=True)
            
        col1_db, col2_db = st.columns([1, 4])
        with col1_db:
            if st.button("🔄 Segarkan Data MySQL", use_container_width=True):
                st.rerun()
    except Exception as e:
        st.write(f"Gangguan pembacaan tabel MySQL: {e}")

